# Pokémon V6 training on Kaggle

CPU-oriented training for the current compact-action V6 policy. Attach the `pokemon-kaggle-v6-training-bundle` dataset before running. A GPU accelerator is not required because the current policy and simulator explicitly train on CPU.

In [ ]:
# Configuration: this is the only cell normally edited.
RUN_KIND = "finetune"          # "finetune" or "base"
DECK_ID = "bank_63"            # Local factory target 15/16: scheduled much later
BASE_ID = "base_b"             # Assigned parent for bank_63 in the selected A/C/B rotation
TIMESTEPS = 400_000              # Use 1_000_000 for RUN_KIND="base"
SEED = 20260785                 # base_b seed 20260722 + deck number 63
NUM_ENVS = 8                     # Reduce to 4 if the runtime becomes unstable
WANDB_PROJECT = "pokemon_kaggle"
RUN_LABEL = f"kaggle_{RUN_KIND}_{DECK_ID}_{BASE_ID}_{TIMESTEPS}"


In [ ]:
from pathlib import Path
import os, platform, shutil, subprocess, sys, time, zipfile

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/pokemon_v6")
bundle_candidates = list(KAGGLE_INPUT.rglob("pokemon_kaggle_v6_bundle.zip"))
project_candidates = [
    path for path in KAGGLE_INPUT.rglob("pokemon_kaggle")
    if (path / "src" / "train.py").is_file()
]
assert bundle_candidates or project_candidates, (
    "Training bundle not found under /kaggle/input. Attach the "
    "pokemon-kaggle-v6-training-bundle dataset first."
)
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
PROJECT = WORK_ROOT / "pokemon_kaggle"
if project_candidates:
    shutil.copytree(project_candidates[0], PROJECT)
else:
    with zipfile.ZipFile(bundle_candidates[0]) as archive:
        archive.extractall(WORK_ROOT)
restored_models = []
for extracted_model in list((PROJECT / "models").rglob("*")):
    if (
        extracted_model.is_dir()
        and (extracted_model / "data").is_file()
        and (extracted_model / "policy.pth").is_file()
    ):
        if extracted_model.suffix == ".zip":
            temporary_base = extracted_model.parent / f"{extracted_model.stem}.restored"
            restored = Path(shutil.make_archive(str(temporary_base), "zip", extracted_model))
            extracted_model.rename(extracted_model.with_name(f"{extracted_model.name}.extracted"))
            restored.rename(extracted_model)
            restored = extracted_model
        else:
            restored = Path(shutil.make_archive(str(extracted_model), "zip", extracted_model))
            extracted_model.rename(extracted_model.with_name(f"{extracted_model.name}.extracted"))
        restored_models.append(restored)
print(f"Restored {len(restored_models)} model ZIP(s) extracted by Kaggle.")
os.chdir(PROJECT)
print({"python": sys.version, "platform": platform.platform(), "cpus": os.cpu_count()})
print("Project:", PROJECT)


In [ ]:
# Internet must be enabled for this dependency setup cell. Versions match the local checkpoints.
import subprocess, sys

if "PROJECT" not in globals():
    raise RuntimeError("Run the project setup cell directly above this one first.")

packages = [
    "stable-baselines3==2.9.0",
    "sb3-contrib==2.9.0",
    "gymnasium==1.2.0",
    "wandb==0.28.0",
    "tensorboard",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

sys.path.insert(0, str(PROJECT))
from src.cg.api import all_card_data
cards = all_card_data()
print(f"Native Linux simulator loaded; {len(cards)} card records available.")


In [ ]:
# Read W&B credentials without printing the secret. Offline mode remains usable without it.
wandb_mode = "offline"
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    wandb_key = (wandb_key or "").strip()
    if len(wandb_key) >= 40:
        os.environ["WANDB_API_KEY"] = wandb_key
        wandb_mode = "online"
    else:
        os.environ.pop("WANDB_API_KEY", None)
        print(f"WANDB_API_KEY has only {len(wandb_key)} characters; using offline mode.")
except Exception as exc:
    os.environ.pop("WANDB_API_KEY", None)
    print("WANDB_API_KEY unavailable; using offline mode:", type(exc).__name__)
os.environ["WANDB_MODE"] = wandb_mode
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_NAME"] = RUN_LABEL
os.environ["PYTHONPATH"] = str(PROJECT)
os.environ["PYTHONUNBUFFERED"] = "1"
print("W&B mode:", wandb_mode)


In [ ]:
BASE_MODELS = {
    "base_a": "models/foundation/ppo_v6_deck_bank_54_base_a.zip",
    "base_b": "models/foundation/ppo_v6_deck_bank_55_base_b.zip",
    "base_c": "models/foundation/ppo_v6_deck_bank_56_base_c.zip",
}
deck_path = PROJECT / "decks" / "deck_bank" / f"{DECK_ID}.csv"
assert deck_path.is_file(), f"Missing deck: {deck_path}"

ARTIFACTS = Path("/kaggle/working/artifacts")
if ARTIFACTS.exists():
    shutil.rmtree(ARTIFACTS)
model_dir = ARTIFACTS / "models"
model_dir.mkdir(parents=True)
output_model = model_dir / f"ppo_v6_deck_{DECK_ID}_{BASE_ID}.zip"

if RUN_KIND == "finetune":
    assert BASE_ID in BASE_MODELS, f"Unknown BASE_ID: {BASE_ID}"
    source_model = PROJECT / BASE_MODELS[BASE_ID]
    assert source_model.is_file(), f"Missing base model: {source_model}"
    shutil.copy2(source_model, output_model)
    aux_coef = "0.0"
    continuation = ["--continue-existing"]
elif RUN_KIND == "base":
    aux_coef = "0.1"
    continuation = []
else:
    raise ValueError("RUN_KIND must be 'finetune' or 'base'")

command = [
    sys.executable, "src/train.py",
    "--deck", str(deck_path.relative_to(PROJECT)),
    "--model-name", str(output_model),
    *continuation,
    "--timesteps", str(TIMESTEPS),
    "--seed", str(SEED),
    "--aux-coef", aux_coef,
    "--policy-version", "v6",
    "--feature-variant", "full",
    "--card-table",
    "--opp-pool", "decks/opponent_factory_v6_development_pool.json",
    "--num-envs", str(NUM_ENVS),
    "--n-steps", "2048",
    "--batch-size", "1024",
    "--n-epochs", "2",
    "--lr", "0.0001",
    "--ent-coef", "0.008",
    "--clip-range", "0.12",
    "--target-kl", "0.03",
    "--belief-actor",
    "--belief-dim", "64",
    "--rotate-perspective",
]
print("Running:", " ".join(command))
started = time.monotonic()
subprocess.run(command, cwd=PROJECT, env=os.environ.copy(), check=True)
elapsed = time.monotonic() - started
assert output_model.is_file(), f"Training finished without model: {output_model}"
completion_marker = output_model.with_suffix(".complete")
completion_marker.touch()
print(f"Finished in {elapsed / 3600:.2f} h; effective {TIMESTEPS / elapsed:.1f} steps/s")


In [ ]:
# Collect everything needed to download or save as Kaggle notebook output.
if (PROJECT / "logs").exists():
    shutil.copytree(PROJECT / "logs", ARTIFACTS / "logs", dirs_exist_ok=True)
for record in model_dir.glob("*.experiment.json"):
    print("Experiment record:", record.name)
wandb_tmp = Path("/tmp/wandb")
if wandb_tmp.exists():
    shutil.copytree(wandb_tmp, ARTIFACTS / "wandb", dirs_exist_ok=True)
archive = shutil.make_archive("/kaggle/working/pokemon_v6_result", "zip", ARTIFACTS)
print("Model:", output_model)
print("Download archive:", archive)
from IPython.display import FileLink, display
display(FileLink(archive))
